# NIFTY 50 Trading Volume Forecasting

Forecast NIFTY 50 trading volume using time-series analysis and Prophet. The project evaluates a chronological 2025 holdout and produces a 90-trading-day forward forecast with uncertainty intervals.

## 1. Imports and Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from prophet import Prophet


## 2. Download NIFTY 50 Data

The analysis uses NIFTY 50 index data (`^NSEI`) and models daily trading volume. Yahoo Finance provides the historical OHLCV series used here.

In [ ]:
data = yf.download(
    '^NSEI',
    start='2019-01-01',
    end=pd.Timestamp.today().strftime('%Y-%m-%d'),
    auto_adjust=False,
    progress=False
)

if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

data = data[['Open', 'High', 'Low', 'Close', 'Volume']].dropna()
data.index = pd.to_datetime(data.index)
data.head()

In [ ]:
print('Observations:', len(data))
print('Start:', data.index.min().date())
print('End:', data.index.max().date())
print('Missing values:', data.isna().sum().sum())

## 3. Exploratory Analysis

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(data.index, data['Volume'])
plt.title('NIFTY 50 Trading Volume')
plt.xlabel('Date')
plt.ylabel('Volume')
plt.show()

In [ ]:
volume_stats = data['Volume'].describe()
volume_stats

In [ ]:
# A log transform helps stabilize the scale when volume varies substantially over time.
data['log_volume'] = np.log1p(data['Volume'])

plt.figure(figsize=(12, 5))
plt.plot(data.index, data['log_volume'])
plt.title('Log-Transformed NIFTY 50 Trading Volume')
plt.xlabel('Date')
plt.ylabel('log(1 + Volume)')
plt.show()

## 4. Chronological Train/Test Split

To avoid look-ahead bias, all observations from 2025 are kept as an out-of-sample test set. The model is trained only on observations available through 2024-12-31.

In [ ]:
train = data.loc[data.index < '2025-01-01'].copy()
test = data.loc[(data.index >= '2025-01-01') & (data.index < '2026-01-01')].copy()

print('Training observations:', len(train))
print('2025 test observations:', len(test))

## 5. Prepare Data for Prophet

Prophet expects columns named `ds` (timestamp) and `y` (target). The model is fit to log-transformed volume and predictions are transformed back to the original volume scale.

In [ ]:
prophet_train = train.reset_index()[['Date', 'log_volume']].rename(
    columns={'Date': 'ds', 'log_volume': 'y'}
)
prophet_train.head()

## 6. Prophet Model

The model includes trend and yearly/weekly seasonal components. Daily seasonality is disabled because the target is observed on trading days rather than every calendar day.

In [ ]:
model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    interval_width=0.90
)
model.fit(prophet_train)

In [ ]:
future_2025 = pd.DataFrame({'ds': test.index})
forecast_2025 = model.predict(future_2025)

pred_2025 = pd.DataFrame({
    'Actual': test['Volume'].values,
    'Predicted': np.expm1(forecast_2025['yhat'].values),
    'Lower': np.expm1(forecast_2025['yhat_lower'].values),
    'Upper': np.expm1(forecast_2025['yhat_upper'].values)
}, index=test.index)

pred_2025.head()

## 7. Out-of-Sample Evaluation

In [ ]:
actual = pred_2025['Actual'].to_numpy()
predicted = pred_2025['Predicted'].to_numpy()

mae = np.mean(np.abs(actual - predicted))
rmse = np.sqrt(np.mean((actual - predicted) ** 2))
mape = np.mean(np.abs((actual - predicted) / np.where(actual == 0, np.nan, actual))) * 100

metrics = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'MAPE'],
    'Value': [mae, rmse, mape]
})
metrics

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(pred_2025.index, pred_2025['Actual'], label='Actual')
plt.plot(pred_2025.index, pred_2025['Predicted'], label='Predicted')
plt.title('NIFTY 50 Trading Volume: 2025 Out-of-Sample Forecast')
plt.xlabel('Date')
plt.ylabel('Volume')
plt.legend()
plt.show()

## 8. Prophet Components

In [ ]:
fig = model.plot_components(model.predict(model.make_future_dataframe(periods=0)))
plt.show()

## 9. Refit on All Available Data

After evaluating the model on the 2025 holdout, the final model is refit using all currently available observations before generating the forward forecast.

In [ ]:
final_train = data.reset_index()[['Date', 'log_volume']].rename(
    columns={'Date': 'ds', 'log_volume': 'y'}
)

final_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    interval_width=0.90
)
final_model.fit(final_train)

## 10. 90-Trading-Day Forward Forecast

Only weekdays are generated and the next 90 dates after the latest observed date are used. This is a calendar approximation to trading days; market holidays are not removed automatically.

In [ ]:
future_dates = pd.bdate_range(
    start=data.index.max() + pd.Timedelta(days=1),
    periods=90
)

future = pd.DataFrame({'ds': future_dates})
future_forecast = final_model.predict(future)

forecast_90 = pd.DataFrame({
    'Date': future_forecast['ds'],
    'Forecast': np.expm1(future_forecast['yhat']),
    'Lower_90': np.expm1(future_forecast['yhat_lower']),
    'Upper_90': np.expm1(future_forecast['yhat_upper'])
}).set_index('Date')

forecast_90.head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(data.index[-250:], data['Volume'].tail(250), label='Historical Volume')
plt.plot(forecast_90.index, forecast_90['Forecast'], label='90-Day Forecast')
plt.fill_between(
    forecast_90.index,
    forecast_90['Lower_90'],
    forecast_90['Upper_90'],
    alpha=0.2,
    label='90% Prediction Interval'
)
plt.title('NIFTY 50 Trading Volume: 90-Trading-Day Forecast')
plt.xlabel('Date')
plt.ylabel('Volume')
plt.legend()
plt.show()

## 11. Key Results

In [ ]:
print('----- 2025 Out-of-Sample Performance -----')
print(f'MAE: {mae:,.0f}')
print(f'RMSE: {rmse:,.0f}')
print(f'MAPE: {mape:.2f}%')
print('\n----- Forward Forecast -----')
print(f'Forecast horizon: {len(forecast_90)} business days')
print(f'Forecast start: {forecast_90.index.min().date()}')
print(f'Forecast end: {forecast_90.index.max().date()}')